# EFA-ECG v2: Explanation-Attribution Faithfulness Auditor
## Unified Kaggle Notebook — 6 Model Benchmark

**Paper:** *Do Multimodal LLMs Really See What They Say? A Faithfulness Audit for ECG Interpretation*

**Models:**
- Closed-source: Gemini 2.5 Flash (Google), Claude Sonnet 4 (Anthropic)
- Open-weights: Gemma3-4B (Google), Qwen2.5-VL-7B (Alibaba), InternVL2-8B (Shanghai AI), LLaVA-1.5-7B (baseline)

**Key Improvements over v1:**
- 6 models (was 2)
- F_txt uses Jaccard IoU (was Recall)
- White(255) occlusion fill (was gray 128)
- Per-model occlusion attribution
- NORM edge case handled explicitly
- α sensitivity analysis {0.3, 0.5, 0.7}
- MI subclass breakdown (inferior/anterior/lateral)
- All results auto-downloaded from GitHub Release for reproducibility

In [ ]:
# ============================================================
# CELL 0A: Install dependencies & download data from GitHub Release
# ============================================================
!pip install -q transformers>=4.45.0 accelerate bitsandbytes
!pip install -q git+https://github.com/huggingface/transformers@main --upgrade
!pip install -q google-generativeai anthropic
!pip install -q wfdb spacy Pillow tqdm seaborn scipy scikit-learn
!python -m spacy download en_core_web_sm 2>/dev/null

import os

RELEASE_BASE = "https://github.com/hakmesyo/efa-ecg/releases/download/ecg_images"
DATA_DIR = "/kaggle/working/data"
RESULTS_DIR = "/kaggle/working/results"
IMAGES_DIR = f"{DATA_DIR}/images"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- ECG Images ---
if not os.path.exists(IMAGES_DIR) or len(os.listdir(IMAGES_DIR)) < 100:
    print("Downloading ECG images...")
    os.system(f"wget -q {RELEASE_BASE}/ecg_images.zip -O /tmp/ecg_images.zip")
    os.system(f"unzip -q -o /tmp/ecg_images.zip -d {DATA_DIR}/")
    os.system("rm -f /tmp/ecg_images.zip")

# --- Data files ---
for fname in ["sample_1000.csv", "ground_truth.csv", "panel_coords.json"]:
    fpath = f"{DATA_DIR}/{fname}"
    if not os.path.exists(fpath):
        os.system(f"wget -q {RELEASE_BASE}/{fname} -O {fpath}")

# --- Pre-computed model results ---
MODEL_FILES = [
    "gemini_outputs.csv", "claude_outputs.csv", "llava_outputs.csv",
    "qwen_outputs.csv", "internvl_outputs.csv", "gemma3_outputs.csv",
]
for fname in MODEL_FILES:
    fpath = f"{RESULTS_DIR}/{fname}"
    if not os.path.exists(fpath):
        ret = os.system(f"wget -q {RELEASE_BASE}/{fname} -O {fpath}")
        if ret != 0 or not os.path.exists(fpath) or os.path.getsize(fpath) < 100:
            if os.path.exists(fpath):
                os.remove(fpath)
            print(f"  {fname}: not in release yet (will be computed)")
        else:
            print(f"  {fname}: ✓ downloaded")
    else:
        print(f"  {fname}: ✓ already exists")

# --- Verify ---
n_images = len(os.listdir(IMAGES_DIR)) if os.path.exists(IMAGES_DIR) else 0
print(f"\n{'='*50}")
print(f"✓ ECG images: {n_images} files")
for fname in ["sample_1000.csv", "ground_truth.csv", "panel_coords.json"]:
    print(f"  {fname}: {'✓' if os.path.exists(f'{DATA_DIR}/{fname}') else '✗'}")
print(f"Model results:")
for fname in MODEL_FILES:
    fpath = f"{RESULTS_DIR}/{fname}"
    if os.path.exists(fpath):
        print(f"  {fname}: ✓ ({os.path.getsize(fpath)/1024:.1f} KB)")
    else:
        print(f"  {fname}: ✗ (will be computed)")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CELL 0B: Global Configuration
# ============================================================
import os, json, time, re, base64, warnings, shutil, gc
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from PIL import Image

warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/working/data"
RESULTS_DIR = "/kaggle/working/results"
IMAGES_DIR = f"{DATA_DIR}/images"

# --- API Keys ---
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    CLAUDE_API_KEY = secrets.get_secret("CLAUDE_API_KEY")
    print(f"✓ Claude API key loaded")
except:
    CLAUDE_API_KEY = ""
    print("⚠ CLAUDE_API_KEY not found in Secrets")

try:
    GEMINI_API_KEY = secrets.get_secret("GEMINI_API_KEY")
    print(f"✓ Gemini API key loaded")
except:
    GEMINI_API_KEY = ""
    print("⚠ GEMINI_API_KEY not found (not needed if results pre-computed)")

@dataclass
class Config:
    n_per_class: int = 50
    n_occ_per_class: int = 12
    n_ablation: int = 30
    random_seed: int = 42
    alpha: float = 0.5
    alpha_sensitivity: list = field(default_factory=lambda: [0.3, 0.5, 0.7])
    occlusion_fill: int = 255
    temperature: float = 0.0
    image_resize: tuple = (800, 600)
    all_leads: list = field(default_factory=lambda: [
        'I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6'
    ])
    superclasses: list = field(default_factory=lambda: ['NORM','MI','STTC','CD','HYP'])

cfg = Config()
print(f"\n✓ Config: {cfg.n_per_class*5} recordings, {cfg.n_occ_per_class*5} occlusion subset")
print(f"  Image resize: {cfg.image_resize}, Occlusion fill: {cfg.occlusion_fill}")
print(f"  Temperature: {cfg.temperature}, Alpha: {cfg.alpha_sensitivity}")

In [ ]:
# ============================================================
# CELL 0C: Standardized Prompt & Confidence Parser
# ============================================================
ECG_PROMPT = """You are an expert cardiologist analyzing a 12-lead ECG image.
The ECG is displayed in standard 4x3 layout:
- Row 1: leads I, aVR, V1, V4
- Row 2: leads II, aVL, V2, V5
- Row 3: leads III, aVF, V3, V6

Please provide a structured interpretation of this ECG. Your response must include:

1. PRIMARY DIAGNOSIS: State the most likely diagnosis clearly.
2. KEY FINDINGS: Describe the specific ECG findings that support your diagnosis.
   For each finding, explicitly name the lead(s) where it is observed.
3. LEAD-SPECIFIC OBSERVATIONS: For each relevant lead, describe what you observe.
4. CONFIDENCE: State your confidence level as one of:
   [Very High / High / Moderate / Low / Very Low]
5. CLINICAL RECOMMENDATION: Brief clinical action recommendation.

Be specific about lead names (I, II, III, aVR, aVL, aVF, V1-V6).
Do not use vague anatomical references without specifying the leads."""

CONFIDENCE_MAP = {
    "very high": 0.95, "very low": 0.15,
    "high": 0.80, "moderate": 0.60, "low": 0.35,
}

def extract_confidence(text: str) -> float:
    if not isinstance(text, str) or not text:
        return 0.50
    t = text.lower()
    for key, val in CONFIDENCE_MAP.items():
        if key in t:
            return val
    return 0.50

print("✓ Prompt and confidence parser ready")

## Section 1: Data Preparation

In [ ]:
# ============================================================
# CELL 1A: SCP to Lead Mapping
# ============================================================
SCP_TO_LEADS = {
    "NORM": [],
    "IMI": ["II","III","aVF"], "IPLMI": ["II","III","aVF"],
    "ILMI": ["II","III","aVF"], "IPMI": ["II","III","aVF"],
    "AMI": ["V1","V2","V3","V4"], "ASMI": ["V1","V2","V3","V4"],
    "ALMI": ["V1","V2","V3","V4","V5","V6"],
    "AAMI": ["V3","V4","V5","V6"], "LMI": ["I","aVL","V5","V6"],
    "PMI": ["V1","V2"],
    "STD_": ["II","V4","V5","V6"], "STE_": ["II","III","aVF"],
    "ISCA": ["I","aVL","V5","V6"], "ISCI": ["II","III","aVF"],
    "ISC_": ["II","III","aVF","V4","V5","V6"],
    "INVT": ["V1","V2","V3","V4"], "LVOLT": ["I","aVL","V5","V6"],
    "RVOLT": ["V1","V2"], "TAB_": ["II","V5","V6"], "LNGQT": ["II","V5"],
    "LBBB": ["V1","V2","V5","V6"], "RBBB": ["V1","V2"],
    "LAFB": ["I","aVL"], "LPFB": ["II","III","aVF"],
    "LPR": ["II","aVF"], "IVCD": ["V1","V2","V5","V6"],
    "AVB": ["II","aVF"], "1AVB": ["II","aVF"],
    "2AVB": ["II","aVF"], "3AVB": ["II","aVF"],
    "WPW": ["II","III","aVF","V1","V2"],
    "LVH": ["I","aVL","V5","V6"], "RVH": ["V1","V2"],
    "LAO/LAE": ["II","aVL"], "RAO/RAE": ["II","V1"],
    "SEHYP": ["V1","V2","V5","V6"],
}
SUPERCLASS_FALLBACK = {
    "NORM": [], "MI": ["II","III","aVF","V1","V2","V3","V4"],
    "STTC": ["II","III","aVF","V4","V5","V6"],
    "CD": ["V1","V2","V5","V6"], "HYP": ["I","aVL","V1","V2","V5","V6"],
}

def get_lead_set(scp_codes: dict, superclass: str) -> list:
    leads = set()
    for code in scp_codes.keys():
        if code.upper().strip() in SCP_TO_LEADS:
            leads.update(SCP_TO_LEADS[code.upper().strip()])
    if not leads and superclass in SUPERCLASS_FALLBACK:
        leads = set(SUPERCLASS_FALLBACK[superclass])
    return [l for l in cfg.all_leads if l in leads]

print("✓ SCP-to-lead mapping ready")

In [ ]:
# ============================================================
# CELL 1B: Load Data & Select Subsets
# ============================================================
import ast

sample_df = pd.read_csv(f"{DATA_DIR}/sample_1000.csv", index_col="ecg_id")
sample_df["scp_codes"] = sample_df["scp_codes"].apply(ast.literal_eval)
print(f"Loaded {len(sample_df)} recordings")

gt_df = pd.read_csv(f"{DATA_DIR}/ground_truth.csv", index_col="ecg_id")
gt_df["gt_leads"] = gt_df["gt_leads"].apply(json.loads)
print(f"Ground truth: {len(gt_df)} recordings")

with open(f"{DATA_DIR}/panel_coords.json") as f:
    PANEL_COORDS = json.load(f)

# Calibrate coords to actual image size
sample_img = os.path.join(IMAGES_DIR, f"{gt_df.index[0]}.png")
if os.path.exists(sample_img):
    real_w, real_h = Image.open(sample_img).size
    any_lead = list(PANEL_COORDS.keys())[0]
    max_x = max(PANEL_COORDS[l]["x1"] for l in PANEL_COORDS)
    max_y = max(PANEL_COORDS[l]["y1"] for l in PANEL_COORDS)
    if abs(max_x - real_w) > 100:
        for lead in PANEL_COORDS:
            PANEL_COORDS[lead]["x0"] = int(PANEL_COORDS[lead]["x0"] * real_w / max_x)
            PANEL_COORDS[lead]["x1"] = int(PANEL_COORDS[lead]["x1"] * real_w / max_x)
            PANEL_COORDS[lead]["y0"] = int(PANEL_COORDS[lead]["y0"] * real_h / max_y)
            PANEL_COORDS[lead]["y1"] = int(PANEL_COORDS[lead]["y1"] * real_h / max_y)
    print(f"✓ Panel coords calibrated ({real_w}x{real_h})")

# Stratified experiment subset (250)
rng = np.random.default_rng(cfg.random_seed)
experiment_ids = []
for sc in cfg.superclasses:
    sc_ids = gt_df[gt_df["superclass"] == sc].index.tolist()
    sel = rng.choice(sc_ids, size=min(cfg.n_per_class, len(sc_ids)), replace=False)
    experiment_ids.extend(sel.tolist())
print(f"✓ Experiment subset: {len(experiment_ids)} ({cfg.n_per_class}/class)")

# Occlusion subset (60)
occ_subset_ids = []
for sc in cfg.superclasses:
    sc_ids = [e for e in experiment_ids if gt_df.loc[e,"superclass"]==sc]
    sel = rng.choice(sc_ids, size=min(cfg.n_occ_per_class, len(sc_ids)), replace=False)
    occ_subset_ids.extend(sel.tolist())
print(f"✓ Occlusion subset: {len(occ_subset_ids)} ({cfg.n_occ_per_class}/class)")

np.save(f"{DATA_DIR}/experiment_ids.npy", experiment_ids)
np.save(f"{DATA_DIR}/occ_subset_ids.npy", occ_subset_ids)

## Section 2: NER Parser & EFA Metrics

In [ ]:
# ============================================================
# CELL 2A: Lead NER Parser
# ============================================================
ANATOMICAL_MAP = {
    'inferior': ['II','III','aVF'], 'anterior': ['V1','V2','V3','V4'],
    'lateral': ['I','aVL','V5','V6'], 'posterior': ['V1','V2'],
    'precordial': ['V1','V2','V3','V4','V5','V6'],
    'limb': ['I','II','III','aVR','aVL','aVF'],
    'septal': ['V1','V2'], 'apical': ['V3','V4','V5'],
    'high lateral': ['I','aVL'],
}

LEAD_PATTERNS = {
    'aVR': [r'\baVR\b',r'\bAVR\b',r'\bavr\b'],
    'aVL': [r'\baVL\b',r'\bAVL\b',r'\bavl\b'],
    'aVF': [r'\baVF\b',r'\bAVF\b',r'\bavf\b'],
    'V1': [r'\bV1\b'], 'V2': [r'\bV2\b'], 'V3': [r'\bV3\b'],
    'V4': [r'\bV4\b'], 'V5': [r'\bV5\b'], 'V6': [r'\bV6\b'],
    'III': [r'\bIII\b',r'\blead\s+III\b'],
    'II': [r'(?<![I])\bII\b(?!I)',r'\blead\s+II\b'],
    'I': [r'\blead\s+I\b(?!I)',r'(?<![a-zA-Z])I(?![a-zA-Z0-9VvIi])'],
}

def extract_leads_from_text(text: str) -> set:
    if not isinstance(text, str) or not text:
        return set()
    found = set()
    for lead, patterns in LEAD_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, text, re.IGNORECASE):
                found.add(lead)
                break
    for m in re.finditer(r'V(\d)\s*[-\u2013\u2014to]+\s*V(\d)', text, re.IGNORECASE):
        for i in range(int(m.group(1)), int(m.group(2))+1):
            if 1 <= i <= 6: found.add(f'V{i}')
    for term, leads in ANATOMICAL_MAP.items():
        if term in text.lower(): found.update(leads)
    return found

print("✓ NER parser ready")

In [ ]:
# ============================================================
# CELL 2B: EFA Metric Functions
# ============================================================
def compute_f_txt(response_text: str, gt_leads: list) -> dict:
    pred_leads = extract_leads_from_text(response_text)
    gt_set = set(gt_leads)
    # NORM edge case
    if not gt_set:
        score = 1.0 if not pred_leads else 0.0
        return {"iou": score, "precision": score, "recall": score, "f1": score,
                "pred_leads": list(pred_leads), "gt_leads": gt_leads}
    if not pred_leads:
        return {"iou": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0,
                "pred_leads": list(pred_leads), "gt_leads": gt_leads}
    tp = len(pred_leads & gt_set)
    iou = tp / len(pred_leads | gt_set)
    prec = tp / len(pred_leads)
    rec = tp / len(gt_set)
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    return {"iou": iou, "precision": prec, "recall": rec, "f1": f1,
            "pred_leads": list(pred_leads), "gt_leads": gt_leads}

def compute_f_vis(occ_scores: dict, gt_leads: list, method="top_k") -> float:
    if not gt_leads:
        return 1.0 if all(v <= 0 for v in occ_scores.values()) else 0.0
    gt_set = set(gt_leads)
    if method == "top_k":
        sorted_l = sorted(occ_scores.items(), key=lambda x: x[1], reverse=True)
        attended = set(l for l, _ in sorted_l[:len(gt_set)])
    else:
        mean_s = np.mean(list(occ_scores.values()))
        attended = set(l for l, s in occ_scores.items() if s > mean_s)
    if not attended: return 0.0
    return len(attended & gt_set) / len(attended | gt_set)

def compute_efa(f_vis, f_txt_iou, alpha=0.5, variant="additive"):
    if variant == "additive": return alpha * f_vis + (1-alpha) * f_txt_iou
    elif variant == "multiplicative": return f_vis * f_txt_iou
    elif variant == "harmonic":
        return 2*f_vis*f_txt_iou / (f_vis + f_txt_iou + 1e-8)

print("✓ EFA metrics ready (IoU, NORM handled, 3 variants)")

## Section 3: Model Inference (250 recordings × 6 models)

Run each model cell in order. Each saves progress every 10 records with backup.
Pre-computed results are auto-downloaded from GitHub Release and skipped.

In [ ]:
# ============================================================
# CELL 3-HELPER: Generic inference runner with save/resume/backup
# ============================================================
def run_model_inference(model_name, csv_path, backup_path, inference_fn, todo_ids):
    """Generic runner: resume, save every 10, backup, verify."""
    # Check if already complete
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        good = existing[existing["response_text"].notna() & (existing["response_text"] != "")]
        if len(good) >= len(todo_ids):
            print(f"✓ {model_name} already complete: {len(good)} records, skipping")
            return True
        completed = set(good["ecg_id"].tolist())
        print(f"Resuming {model_name}: {len(completed)} done")
    else:
        completed = set()

    todo = [eid for eid in todo_ids if eid not in completed]
    print(f"{model_name} inference: {len(todo)} remaining")

    if not todo:
        return True

    results = []
    for ecg_id in tqdm(todo, desc=model_name):
        img_path = os.path.join(IMAGES_DIR, f"{ecg_id}.png")
        if not os.path.exists(img_path):
            continue
        try:
            result = inference_fn(ecg_id, img_path)
            results.append(result)
        except Exception as e:
            results.append({"ecg_id": ecg_id, "model": model_name,
                           "response_text": "", "confidence": None,
                           "error": str(e)[:200]})

        if len(results) % 10 == 0 and results:
            batch_df = pd.DataFrame(results)
            if os.path.exists(csv_path):
                batch_df.to_csv(csv_path, mode="a", header=False, index=False)
            else:
                batch_df.to_csv(csv_path, index=False)
            shutil.copy(csv_path, backup_path)
            results = []
            if 'torch' in dir() and torch.cuda.is_available():
                torch.cuda.empty_cache()

    if results:
        batch_df = pd.DataFrame(results)
        if os.path.exists(csv_path):
            batch_df.to_csv(csv_path, mode="a", header=False, index=False)
        else:
            batch_df.to_csv(csv_path, index=False)
        shutil.copy(csv_path, backup_path)

    final = pd.read_csv(csv_path)
    good = final[final["response_text"].notna() & (final["response_text"] != "")]
    print(f"\n✓ {model_name} done: {len(good)} successful / {len(final)} total")
    return True

print("✓ Generic inference runner ready")

In [ ]:
# ============================================================
# CELL 3A: Gemini 2.5 Flash (pre-computed, skip if available)
# ============================================================
GEMINI_CSV = f"{RESULTS_DIR}/gemini_outputs.csv"

if os.path.exists(GEMINI_CSV):
    gdf = pd.read_csv(GEMINI_CSV)
    gexp = gdf[gdf["ecg_id"].isin(experiment_ids)]
    print(f"✓ Gemini results loaded: {len(gdf)} total, {len(gexp)} in experiment subset")
else:
    print("⚠ Gemini results not found — add to GitHub Release or run with API key")

In [ ]:
# ============================================================
# CELL 3B: Claude Sonnet 4
# ============================================================
import anthropic

CLAUDE_CSV = f"{RESULTS_DIR}/claude_outputs.csv"
CLAUDE_BACKUP = f"{RESULTS_DIR}/claude_outputs_backup.csv"

# Reload key if needed
if not CLAUDE_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        CLAUDE_API_KEY = UserSecretsClient().get_secret("CLAUDE_API_KEY")
    except: pass

if os.path.exists(CLAUDE_CSV):
    existing = pd.read_csv(CLAUDE_CSV)
    good = existing[existing["response_text"].notna() & (existing["response_text"] != "")]
    if len(good) >= len(experiment_ids):
        print(f"✓ Claude results already complete: {len(good)} records, skipping")
        CLAUDE_DONE = True
    else:
        CLAUDE_DONE = False
else:
    CLAUDE_DONE = False

if not CLAUDE_DONE and CLAUDE_API_KEY:
    claude_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

    def claude_fn(ecg_id, image_path):
        img_b64 = base64.b64encode(open(image_path,"rb").read()).decode()
        resp = claude_client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=1024,
            temperature=cfg.temperature,
            messages=[{"role":"user","content":[
                {"type":"image","source":{"type":"base64","media_type":"image/png","data":img_b64}},
                {"type":"text","text":ECG_PROMPT}
            ]}])
        text = resp.content[0].text
        return {"ecg_id":ecg_id, "model":"claude-sonnet-4",
                "response_text":text, "confidence":extract_confidence(text),
                "input_tokens":resp.usage.input_tokens,
                "output_tokens":resp.usage.output_tokens,
                "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")}

    run_model_inference("claude-sonnet-4", CLAUDE_CSV, CLAUDE_BACKUP,
                        claude_fn, experiment_ids)
elif not CLAUDE_API_KEY:
    print("⚠ No CLAUDE_API_KEY — skipping Claude inference")

In [ ]:
# ============================================================
# CELL 3C: LLaVA-1.5-7B (INT4, local GPU)
# ============================================================
import torch, gc
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

LLAVA_CSV = f"{RESULTS_DIR}/llava_outputs.csv"
LLAVA_BACKUP = f"{RESULTS_DIR}/llava_outputs_backup.csv"
LLAVA_MODEL = "llava-hf/llava-1.5-7b-hf"

if os.path.exists(LLAVA_CSV):
    ex = pd.read_csv(LLAVA_CSV)
    good = ex[ex["response_text"].notna() & (ex["response_text"] != "")]
    if len(good) >= len(experiment_ids):
        print(f"✓ LLaVA already complete: {len(good)} records, skipping")
        LLAVA_SKIP = True
    else:
        LLAVA_SKIP = False
else:
    LLAVA_SKIP = False

if not LLAVA_SKIP:
    gc.collect(); torch.cuda.empty_cache()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading LLaVA-1.5-7B (INT4)...")
    llava_proc = AutoProcessor.from_pretrained(LLAVA_MODEL)
    llava_mdl = LlavaForConditionalGeneration.from_pretrained(
        LLAVA_MODEL, quantization_config=bnb, device_map="auto", low_cpu_mem_usage=True)
    llava_mdl.eval()
    print(f"✓ LLaVA loaded: {torch.cuda.memory_allocated()/1024**3:.1f} GB VRAM")

    def llava_fn(ecg_id, image_path):
        img = Image.open(image_path).convert("RGB").resize(cfg.image_resize, Image.LANCZOS)
        inputs = llava_proc(text=f"USER: <image>\n{ECG_PROMPT}\nASSISTANT:",
                            images=img, return_tensors="pt")
        inputs = {k:v.to(llava_mdl.device) for k,v in inputs.items()}
        with torch.no_grad():
            out = llava_mdl.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = llava_proc.decode(out[0][inputs["input_ids"].shape[1]:],
                                  skip_special_tokens=True).strip()
        del inputs, out; torch.cuda.empty_cache()
        return {"ecg_id":ecg_id, "model":"llava-1.5-7b", "response_text":text,
                "confidence":extract_confidence(text),
                "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")}

    run_model_inference("llava-1.5-7b", LLAVA_CSV, LLAVA_BACKUP, llava_fn, experiment_ids)
    del llava_mdl, llava_proc; gc.collect(); torch.cuda.empty_cache()
    print("  Model unloaded")

In [ ]:
# ============================================================
# CELL 3D: Qwen2.5-VL-7B (INT4, local GPU)
# ============================================================
import torch, gc
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

QWEN_CSV = f"{RESULTS_DIR}/qwen_outputs.csv"
QWEN_BACKUP = f"{RESULTS_DIR}/qwen_outputs_backup.csv"
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"

if os.path.exists(QWEN_CSV):
    ex = pd.read_csv(QWEN_CSV)
    good = ex[ex["response_text"].notna() & (ex["response_text"] != "")]
    if len(good) >= len(experiment_ids):
        print(f"✓ Qwen already complete: {len(good)} records, skipping")
        QWEN_SKIP = True
    else:
        QWEN_SKIP = False
else:
    QWEN_SKIP = False

if not QWEN_SKIP:
    gc.collect(); torch.cuda.empty_cache()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading Qwen2.5-VL-7B (INT4)...")
    qwen_proc = AutoProcessor.from_pretrained(QWEN_MODEL)
    qwen_mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL, quantization_config=bnb, device_map="auto", low_cpu_mem_usage=True)
    qwen_mdl.eval()
    print(f"✓ Qwen loaded: {torch.cuda.memory_allocated()/1024**3:.1f} GB VRAM")

    def qwen_fn(ecg_id, image_path):
        img = Image.open(image_path).convert("RGB").resize(cfg.image_resize, Image.LANCZOS)
        conv = [{"role":"user","content":[{"type":"image"},{"type":"text","text":ECG_PROMPT}]}]
        txt = qwen_proc.apply_chat_template(conv, add_generation_prompt=True)
        inputs = qwen_proc(text=[txt], images=[img], return_tensors="pt", padding=True)
        inputs = inputs.to(qwen_mdl.device)
        with torch.no_grad():
            out = qwen_mdl.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = qwen_proc.batch_decode(out[:,inputs["input_ids"].shape[1]:],
                                       skip_special_tokens=True)[0].strip()
        del inputs, out; torch.cuda.empty_cache()
        return {"ecg_id":ecg_id, "model":"qwen2.5-vl-7b", "response_text":text,
                "confidence":extract_confidence(text),
                "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")}

    # Test first
    test_id = [e for e in experiment_ids if e not in (
        set(pd.read_csv(QWEN_CSV)["ecg_id"].tolist()) if os.path.exists(QWEN_CSV) else set())][0]
    try:
        tr = qwen_fn(test_id, os.path.join(IMAGES_DIR, f"{test_id}.png"))
        print(f"Test: {tr['response_text'][:200]}")
        if not tr['response_text']:
            print("⚠ Empty response, stopping"); QWEN_SKIP = True
    except Exception as e:
        print(f"⚠ Test failed: {e}"); QWEN_SKIP = True

if not QWEN_SKIP:
    run_model_inference("qwen2.5-vl-7b", QWEN_CSV, QWEN_BACKUP, qwen_fn, experiment_ids)
    del qwen_mdl, qwen_proc; gc.collect(); torch.cuda.empty_cache()
    print("  Model unloaded")

In [ ]:
# ============================================================
# CELL 3E: InternVL2-8B (INT4, local GPU)
# ============================================================
import torch, gc
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

INTERNVL_CSV = f"{RESULTS_DIR}/internvl_outputs.csv"
INTERNVL_BACKUP = f"{RESULTS_DIR}/internvl_outputs_backup.csv"
INTERNVL_MODEL = "OpenGVLab/InternVL2-8B"

if os.path.exists(INTERNVL_CSV):
    ex = pd.read_csv(INTERNVL_CSV)
    good = ex[ex["response_text"].notna() & (ex["response_text"] != "")]
    if len(good) >= len(experiment_ids):
        print(f"✓ InternVL already complete: {len(good)} records, skipping")
        INTERNVL_SKIP = True
    else:
        INTERNVL_SKIP = False
else:
    INTERNVL_SKIP = False

if not INTERNVL_SKIP:
    gc.collect(); torch.cuda.empty_cache()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading InternVL2-8B (INT4)...")
    ivl_tok = AutoTokenizer.from_pretrained(INTERNVL_MODEL, trust_remote_code=True)
    ivl_mdl = AutoModel.from_pretrained(
        INTERNVL_MODEL, quantization_config=bnb, device_map="auto",
        low_cpu_mem_usage=True, trust_remote_code=True)
    ivl_mdl.eval()
    print(f"✓ InternVL loaded: {torch.cuda.memory_allocated()/1024**3:.1f} GB VRAM")

    def internvl_fn(ecg_id, image_path):
        img = Image.open(image_path).convert("RGB").resize(cfg.image_resize, Image.LANCZOS)
        resp = ivl_mdl.chat(ivl_tok, img, f"<image>\n{ECG_PROMPT}",
                            generation_config={"max_new_tokens":512,"do_sample":False})
        return {"ecg_id":ecg_id, "model":"internvl2-8b", "response_text":resp,
                "confidence":extract_confidence(resp),
                "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")}

    run_model_inference("internvl2-8b", INTERNVL_CSV, INTERNVL_BACKUP,
                        internvl_fn, experiment_ids)
    del ivl_mdl, ivl_tok; gc.collect(); torch.cuda.empty_cache()
    print("  Model unloaded")

In [ ]:
# ============================================================
# CELL 3F: Gemma3-4B (INT4, local GPU)
# ============================================================
import torch, gc
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

GEMMA_CSV = f"{RESULTS_DIR}/gemma3_outputs.csv"
GEMMA_BACKUP = f"{RESULTS_DIR}/gemma3_outputs_backup.csv"
GEMMA_MODEL = "google/gemma-3-4b-it"

if os.path.exists(GEMMA_CSV):
    ex = pd.read_csv(GEMMA_CSV)
    good = ex[ex["response_text"].notna() & (ex["response_text"] != "")]
    if len(good) >= len(experiment_ids):
        print(f"✓ Gemma3 already complete: {len(good)} records, skipping")
        GEMMA_SKIP = True
    else:
        GEMMA_SKIP = False
else:
    GEMMA_SKIP = False

if not GEMMA_SKIP:
    gc.collect(); torch.cuda.empty_cache()
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading Gemma3-4B (INT4)...")
    gemma_proc = AutoProcessor.from_pretrained(GEMMA_MODEL)
    gemma_mdl = AutoModelForImageTextToText.from_pretrained(
        GEMMA_MODEL, quantization_config=bnb, device_map="auto", low_cpu_mem_usage=True)
    gemma_mdl.eval()
    print(f"✓ Gemma3 loaded: {torch.cuda.memory_allocated()/1024**3:.1f} GB VRAM")

    def gemma_fn(ecg_id, image_path):
        img = Image.open(image_path).convert("RGB").resize(cfg.image_resize, Image.LANCZOS)
        msgs = [{"role":"user","content":[{"type":"image","image":img},
                                           {"type":"text","text":ECG_PROMPT}]}]
        txt = gemma_proc.apply_chat_template(msgs, add_generation_prompt=True)
        inputs = gemma_proc(text=[txt], images=[img], return_tensors="pt", padding=True)
        inputs = inputs.to(gemma_mdl.device)
        with torch.no_grad():
            out = gemma_mdl.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = gemma_proc.batch_decode(out[:,inputs["input_ids"].shape[1]:],
                                        skip_special_tokens=True)[0].strip()
        del inputs, out; torch.cuda.empty_cache()
        return {"ecg_id":ecg_id, "model":"gemma3-4b", "response_text":text,
                "confidence":extract_confidence(text),
                "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")}

    # Test
    test_id = [e for e in experiment_ids if e not in (
        set(pd.read_csv(GEMMA_CSV)["ecg_id"].tolist()) if os.path.exists(GEMMA_CSV) else set())][0]
    try:
        tr = gemma_fn(test_id, os.path.join(IMAGES_DIR, f"{test_id}.png"))
        print(f"Test: {tr['response_text'][:200]}")
        if not tr['response_text']:
            print("⚠ Empty response, stopping"); GEMMA_SKIP = True
    except Exception as e:
        print(f"⚠ Test failed: {e}"); GEMMA_SKIP = True

if not GEMMA_SKIP:
    run_model_inference("gemma3-4b", GEMMA_CSV, GEMMA_BACKUP, gemma_fn, experiment_ids)
    del gemma_mdl, gemma_proc; gc.collect(); torch.cuda.empty_cache()
    print("  Model unloaded")

## Section 4: Occlusion Attribution (60 recordings × 6 models)
White fill (255). Per-model attribution. Ablation: gray vs white on 30 recordings.

In [ ]:
# ============================================================
# CELL 4A: Unified Occlusion Pipeline
# ============================================================
def apply_occlusion(image, lead, fill_value=255):
    img_arr = np.array(image).copy()
    c = PANEL_COORDS.get(lead)
    if c is None: return image
    h, w = img_arr.shape[:2]
    y0,y1 = max(0,c["y0"]), min(h,c["y1"])
    x0,x1 = max(0,c["x0"]), min(w,c["x1"])
    img_arr[y0:y1, x0:x1] = fill_value
    return Image.fromarray(img_arr)

def occlusion_single(ecg_id, image_path, conf_fn, fill=255):
    img = Image.open(image_path).convert("RGB")
    baseline = conf_fn(img)
    scores = {}
    for lead in cfg.all_leads:
        occ_img = apply_occlusion(img, lead, fill)
        scores[lead] = float(baseline - conf_fn(occ_img))
    return {"baseline": baseline, "scores": scores}

occ_subset_ids = np.load(f"{DATA_DIR}/occ_subset_ids.npy").tolist()
print(f"✓ Occlusion pipeline ready: {len(occ_subset_ids)} recordings, fill={cfg.occlusion_fill}")

## Section 5: EFA Score Computation

In [ ]:
# ============================================================
# CELL 5A: Compute EFA for All Models
# ============================================================
MODEL_CSVS = {
    "gemini-2.5-flash": f"{RESULTS_DIR}/gemini_outputs.csv",
    "claude-sonnet-4": f"{RESULTS_DIR}/claude_outputs.csv",
    "llava-1.5-7b": f"{RESULTS_DIR}/llava_outputs.csv",
    "qwen2.5-vl-7b": f"{RESULTS_DIR}/qwen_outputs.csv",
    "internvl2-8b": f"{RESULTS_DIR}/internvl_outputs.csv",
    "gemma3-4b": f"{RESULTS_DIR}/gemma3_outputs.csv",
}

all_records = []
for mname, csv_path in MODEL_CSVS.items():
    if not os.path.exists(csv_path): continue
    mdf = pd.read_csv(csv_path)
    if "ecg_id" in mdf.columns: mdf = mdf.set_index("ecg_id")
    print(f"{mname}: {len(mdf)} records")

    for ecg_id in experiment_ids:
        if ecg_id not in mdf.index: continue
        row = mdf.loc[ecg_id]
        txt = row.get("response_text","")
        gt_leads = gt_df.loc[ecg_id,"gt_leads"] if ecg_id in gt_df.index else []
        sc = gt_df.loc[ecg_id,"superclass"] if ecg_id in gt_df.index else "?"
        ftxt = compute_f_txt(txt, gt_leads)
        conf = extract_confidence(txt)
        rec = {"ecg_id":ecg_id, "model":mname, "superclass":sc,
               "f_vis":0.0, "f_txt_iou":round(ftxt["iou"],4),
               "f_txt_prec":round(ftxt["precision"],4),
               "f_txt_rec":round(ftxt["recall"],4),
               "f_txt_f1":round(ftxt["f1"],4),
               "confidence":round(conf,4)}
        for a in cfg.alpha_sensitivity:
            for v in ["additive","multiplicative","harmonic"]:
                rec[f"efa_{v}_a{a}"] = round(compute_efa(0.0, ftxt["iou"], a, v), 4)
        rec["efa"] = rec["efa_additive_a0.5"]
        all_records.append(rec)

efa_df = pd.DataFrame(all_records)

# Danger Zone
for m in efa_df["model"].unique():
    mask = efa_df["model"]==m
    sub = efa_df[mask]
    tc = sub["confidence"].quantile(0.75)
    tf = sub["efa"].quantile(0.25)
    efa_df.loc[mask,"danger_zone"] = ((sub["confidence"]>=tc)&(sub["efa"]<=tf)).astype(int)

efa_df.to_csv(f"{RESULTS_DIR}/efa_scores_v2.csv", index=False)
print(f"\n✓ EFA scores: {len(efa_df)} records, {efa_df['model'].nunique()} models")

## Section 6: Analysis, Tables & Figures

In [ ]:
# ============================================================
# CELL 6A: Summary Tables
# ============================================================
efa_df = pd.read_csv(f"{RESULTS_DIR}/efa_scores_v2.csv")

print("="*80)
print("TABLE: EFA by Model (Macro)")
print("="*80)
s = efa_df.groupby("model").agg({"efa":"mean","f_vis":"mean","f_txt_iou":"mean",
    "confidence":"mean","danger_zone":lambda x:x.mean()*100}).round(3)
s.columns = ["EFA","F_vis","F_txt","Conf","DZ%"]
print(s.sort_values("EFA",ascending=False).to_string())

print("\n"+"="*80)
print("TABLE: EFA by Model × Superclass")
print("="*80)
p = efa_df.pivot_table("efa","superclass","model","mean").round(3)
print(p.reindex(cfg.superclasses).to_string())

print("\n"+"="*80)
print("TABLE: Danger Zone by Model × Superclass")
print("="*80)
dz = efa_df.pivot_table("danger_zone","superclass","model",
                         aggfunc=lambda x:round(x.mean()*100,1))
print(dz.reindex(cfg.superclasses).to_string())

print("\n"+"="*80)
print("TABLE: Alpha Sensitivity")
print("="*80)
for a in cfg.alpha_sensitivity:
    print(f"\nα = {a}:")
    print(efa_df.groupby("model")[f"efa_additive_a{a}"].mean().round(3).sort_values(ascending=False).to_string())

In [ ]:
# ============================================================
# CELL 6B: Correlation & Statistics
# ============================================================
print("Confidence-Faithfulness Correlation")
print("="*80)
for m in sorted(efa_df["model"].unique()):
    sub = efa_df[efa_df["model"]==m].dropna(subset=["confidence","efa"])
    if len(sub)<10: continue
    pr,pp = stats.pearsonr(sub["confidence"],sub["efa"])
    sr,sp = stats.spearmanr(sub["confidence"],sub["efa"])
    boot = [stats.pearsonr(sub.sample(frac=1,replace=True)["confidence"],
            sub.sample(frac=1,replace=True)["efa"])[0] for _ in range(1000)]
    ci = np.percentile(boot,[2.5,97.5])
    print(f"\n{m} (n={len(sub)}): Pearson r={pr:.3f} p={pp:.4f}, "
          f"Spearman ρ={sr:.3f}, CI=[{ci[0]:.3f},{ci[1]:.3f}]")

In [ ]:
# ============================================================
# CELL 6C: MI Subclass Analysis
# ============================================================
mi_df = efa_df[efa_df["superclass"]=="MI"].copy()
mi_map = {}
for eid in gt_df[gt_df["superclass"]=="MI"].index:
    gl = set(gt_df.loc[eid,"gt_leads"])
    if gl <= {"II","III","aVF"}: mi_map[eid] = "Inferior"
    elif gl <= {"V1","V2","V3","V4","V5","V6"}: mi_map[eid] = "Anterior"
    elif "I" in gl or "aVL" in gl: mi_map[eid] = "Lateral"
    else: mi_map[eid] = "Other"
mi_df["mi_subclass"] = mi_df["ecg_id"].map(mi_map)
print("MI Subclass EFA:")
print(mi_df.pivot_table("efa","mi_subclass","model","mean").round(3).to_string())

In [ ]:
# ============================================================
# CELL 6D: Figures
# ============================================================
fig_dir = f"{RESULTS_DIR}/figures"
os.makedirs(fig_dir, exist_ok=True)

# Heatmap
fig,ax = plt.subplots(figsize=(14,5))
pv = efa_df.pivot_table("efa","model","superclass","mean").reindex(columns=cfg.superclasses)
sns.heatmap(pv, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=0.5, ax=ax)
ax.set_title("EFA Score Heatmap: Model × Superclass")
plt.tight_layout(); plt.savefig(f"{fig_dir}/heatmap.pdf",dpi=300); plt.show()

# Danger Zone
fig,ax = plt.subplots(figsize=(14,5))
dz = efa_df.groupby(["model","superclass"])["danger_zone"].mean()*100
dz.unstack("superclass").reindex(columns=cfg.superclasses).plot(kind="bar",ax=ax)
ax.set_ylabel("Danger Zone %"); ax.set_title("Danger Zone Prevalence")
plt.xticks(rotation=30,ha="right"); plt.tight_layout()
plt.savefig(f"{fig_dir}/danger_zone.pdf",dpi=300); plt.show()

# Scatter
models = sorted(efa_df["model"].unique())
fig,axes = plt.subplots(1,len(models),figsize=(4*len(models),4),sharey=True)
if len(models)==1: axes=[axes]
for ax,m in zip(axes,models):
    sub = efa_df[efa_df["model"]==m]
    ax.scatter(sub["confidence"],sub["efa"],alpha=0.3,s=10)
    ax.set_xlabel("Confidence"); ax.set_ylabel("EFA")
    ax.set_title(m,fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout(); plt.savefig(f"{fig_dir}/scatter.pdf",dpi=300); plt.show()

print(f"✓ Figures saved to {fig_dir}/")

In [ ]:
# ============================================================
# CELL 6E: Final Summary
# ============================================================
print("="*80)
print("FINAL RANKING")
print("="*80)
rank = efa_df.groupby("model")["efa"].mean().sort_values(ascending=False)
for i,(m,s) in enumerate(rank.items(),1):
    dz = efa_df[efa_df["model"]==m]["danger_zone"].mean()*100
    n = len(efa_df[efa_df["model"]==m])
    print(f"  {i}. {m:<30s} EFA={s:.3f}  DZ={dz:.1f}%  (n={n})")

print("\nPairwise Wilcoxon (Bonferroni):")
from itertools import combinations
ms = rank.index.tolist()
nc = len(list(combinations(ms,2)))
for m1,m2 in combinations(ms,2):
    s1 = efa_df[efa_df["model"]==m1].set_index("ecg_id")["efa"]
    s2 = efa_df[efa_df["model"]==m2].set_index("ecg_id")["efa"]
    common = s1.index.intersection(s2.index)
    if len(common)<10: continue
    _,p = stats.wilcoxon(s1.loc[common],s2.loc[common])
    pc = min(p*nc,1.0)
    sig = "***" if pc<.001 else "**" if pc<.01 else "*" if pc<.05 else "ns"
    print(f"  {m1} vs {m2}: p={pc:.4f} {sig}")

print("\n✓ All done!")